In [ ]:
!pip install transformers==4.44.0 seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import requests, os
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset, DatasetDict
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score

print("All imports OK")

os.makedirs("/content/bc5cdr", exist_ok=True)
base = "https://raw.githubusercontent.com/cambridgeltl/MTL-Bioinformatics-2016/master/data/BC5CDR-chem-IOB/"
urls = {"train": base+"train.tsv", "dev": base+"devel.tsv", "test": base+"test.tsv"}
for split, url in urls.items():
    r = requests.get(url)
    with open(f"/content/bc5cdr/{split}.txt", "w") as f:
        f.write(r.text)
    print(f"Downloaded {split}: {len(r.text.strip().split(chr(10)))} lines")


All imports OK
Downloaded train: 122729 lines
Downloaded dev: 122033 lines
Downloaded test: 129546 lines


In [ ]:
def parse_conll(filepath):
    sentences, labels = [], []
    tokens, tags = [], []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    tags.append(parts[-1])
    if tokens:
        sentences.append(tokens)
        labels.append(tags)
    return sentences, labels

train_sents, train_tags = parse_conll("/content/bc5cdr/train.txt")
dev_sents, dev_tags = parse_conll("/content/bc5cdr/dev.txt")
test_sents, test_tags = parse_conll("/content/bc5cdr/test.txt")

all_tags = set(t for tags in train_tags+dev_tags+test_tags for t in tags)
label_list = sorted(all_tags)
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Train: {len(train_sents)} | Dev: {len(dev_sents)} | Test: {len(test_sents)}")
print(f"Labels: {label_list}")

def make_dataset(sentences, tags):
    return Dataset.from_dict({
        "tokens": sentences,
        "tags": [[label2id[t] for t in tag_seq] for tag_seq in tags]
    })

dataset = DatasetDict({
    "train": make_dataset(train_sents, train_tags),
    "validation": make_dataset(dev_sents, dev_tags),
    "test": make_dataset(test_sents, test_tags),
})
print(dataset)


Train: 4560 | Dev: 4581 | Test: 4797
Labels: ['B-Chemical', 'I-Chemical', 'O']
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 4560
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 4581
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 4797
    })
})


In [ ]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned, prev_word_id = [], None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != prev_word_id:
                aligned.append(labels[word_id])
            else:
                aligned.append(-100)
            prev_word_id = word_id
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)
print("Tokenization done")
print(tokenized_dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/4560 [00:00<?, ? examples/s]

Map:   0%|          | 0/4581 [00:00<?, ? examples/s]

Map:   0%|          | 0/4797 [00:00<?, ? examples/s]

Tokenization done
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4560
    })
    validation: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4581
    })
    test: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4797
    })
})


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print(f"Model loaded — {len(label_list)} labels")


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded — 3 labels


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        true_l, true_p = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_l.append(id2label[l])
                true_p.append(id2label[p])
        true_labels.append(true_l)
        true_preds.append(true_p)
    return {
        "f1": f1_score(true_labels, true_preds),
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
    }

training_args = TrainingArguments(
    output_dir="./biobert-bc5cdr-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.030200,0.031424,0.919671,0.916077,0.923293
2,0.012500,0.033427,0.925721,0.927027,0.924419
3,0.004900,0.039000,0.925926,0.932978,0.918980
4,0.002400,0.041993,0.924902,0.921288,0.928545


TrainOutput(global_step=1140, training_loss=0.02328487447087179, metrics={'train_runtime': 309.1308, 'train_samples_per_second': 59.004, 'train_steps_per_second': 3.688, 'total_flos': 1191523983114240.0, 'train_loss': 0.02328487447087179, 'epoch': 4.0})

In [ ]:
results = trainer.evaluate(tokenized_dataset["test"])
print("\n=== TEST SET RESULTS ===")
print(f"F1: {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall: {results['eval_recall']:.4f}")



=== TEST SET RESULTS ===
F1: 0.9062
Precision: 0.9180
Recall: 0.8948


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE_PATH = "/content/drive/MyDrive/biobert_bc5cdr_ner"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Saved to {SAVE_PATH}")

Mounted at /content/drive
Saved to /content/drive/MyDrive/biobert_bc5cdr_ner
